### 1. Подготовка данных и базовая модель

In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score

# 1-2. Загрузка данных и разделение на X, y
california = fetch_california_housing()
X = pd.DataFrame(california.data, columns=california.feature_names)
y = california.target

# 3. Разделение на train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 4. Масштабирование
scaler_base = StandardScaler()
X_train_scaled = scaler_base.fit_transform(X_train)
X_test_scaled = scaler_base.transform(X_test)

# 5. Обучение базовой линейной регрессии
lr_base = LinearRegression()
lr_base.fit(X_train_scaled, y_train)

r2_base_test = r2_score(y_test, lr_base.predict(X_test_scaled))
print(f"Базовый R2 на тестовой выборке: {r2_base_test:.4f}")

Базовый R2 на тестовой выборке: 0.5958


### 2. Добавляем шум

In [ ]:
np.random.seed(42)

# 1. Генерируем 30 столбцов случайного шума
noise_train = np.random.normal(0, 10, size=(X_train.shape[0], 30))
noise_test = np.random.normal(0, 10, size=(X_test.shape[0], 30))

# Приклеиваем шум к исходным данным
X_train_noisy = np.hstack([X_train, noise_train])
X_test_noisy = np.hstack([X_test, noise_test])

print(f"Размер старых данных: {X_train.shape[1]} признаков")
print(f"Размер новых (зашумленных) данных: {X_train_noisy.shape[1]} признаков")

# 2. Повторное масштабирование новых данных
scaler_noisy = StandardScaler()
X_train_noisy_scaled = scaler_noisy.fit_transform(X_train_noisy)
X_test_noisy_scaled = scaler_noisy.transform(X_test_noisy)

Размер старых данных: 8 признаков
Размер новых (зашумленных) данных: 38 признаков


### 3. Демонстрация поломки

In [3]:
# 1. Обучение новой LinearRegression на зашумленных данных
lr_noisy = LinearRegression()
lr_noisy.fit(X_train_noisy_scaled, y_train)

# 2. Расчет R2 для train и test
r2_noisy_train = r2_score(y_train, lr_noisy.predict(X_train_noisy_scaled))
r2_noisy_test = r2_score(y_test, lr_noisy.predict(X_test_noisy_scaled))

print(f"R2 на зашумленном TRAIN: {r2_noisy_train:.4f}")
print(f"R2 на зашумленном TEST:  {r2_noisy_test:.4f}")

# 3. Сравнение с базовым
print(f"Падение качества на тесте (Базовый R2 - Зашумленный R2): {r2_base_test - r2_noisy_test:.4f}")


R2 на зашумленном TRAIN: 0.6099
R2 на зашумленном TEST:  0.5950
Падение качества на тесте (Базовый R2 - Зашумленный R2): 0.0008


### 4. Поиск и спасение (Ridge)

In [7]:
# 1. Задаем сетку alpha
alphas = [0.1, 1, 10, 100, 1000]

best_alpha = None
best_r2_ridge = -float('inf')

print("Результаты Ridge-регрессии на зашумленном test:")
for alpha in alphas:
    # Обучение Ridge-регрессии
    ridge_model = Ridge(alpha=alpha)
    ridge_model.fit(X_train_noisy_scaled, y_train)
    
    # Предсказание и расчет R2 только на TEST (зашумленном)
    r2_ridge_test = r2_score(y_test, ridge_model.predict(X_test_noisy_scaled))
    print(f"Alpha = {alpha:^5} | R2_test = {r2_ridge_test:.4f}")
    
    # 4. Поиск наилучшего alpha
    if r2_ridge_test > best_r2_ridge:
        best_r2_ridge = r2_ridge_test
        best_alpha = alpha

print(f"\nЛучшее значение alpha: {best_alpha}")
print(f"Лучший R2 на тесте:    {best_r2_ridge:.4f}")


Результаты Ridge-регрессии на зашумленном test:
Alpha =  0.1  | R2_test = 0.5950
Alpha =   1   | R2_test = 0.5950
Alpha =  10   | R2_test = 0.5952
Alpha =  100  | R2_test = 0.5960
Alpha = 1000  | R2_test = 0.5784

Лучшее значение alpha: 100
Лучший R2 на тесте:    0.5960


### 5. Результаты

In [6]:
import pandas as pd

# Создаем итоговую таблицу
summary_df = pd.DataFrame({
    'Модель / условия': [
        'Базовая (предыдущая практика)',
        'Сломанная (LinearRegression + шум)',
        f'Исправленная (Ridge + шум, alpha = {best_alpha})'
    ],
    'R2 на test': [
        round(r2_base_test, 4),
        round(r2_noisy_test, 4),
        round(best_r2_ridge, 4)
    ],
    'Вывод': [
        'Наш ориентир',
        'Катастрофа!',
        'Спасение!'
    ]
})

display(summary_df)

# Ответы на вопросы
difference_drop = r2_base_test - r2_noisy_test
difference_recovery = r2_base_test - best_r2_ridge

print("\n Вопросы: ")
print(f"2. Насколько упало качество (R2) после добавления шума? На {difference_drop:.4f} единиц.")
print(f"3. При каком alpha Ridge показал лучший результат? При alpha = {best_alpha}")
print(f"4. Удалось ли вернуть качество к базовому уровню? Да, разрыв с базовой моделью составляет всего {difference_recovery:.4f}. Ridge обнулил влияние 30 шумовых признаков.")


,Модель / условия,R2 на test,Вывод
0,Базовая (предыдущая практика),0.5958,Наш ориентир
1,Сломанная (LinearRegression + шум),0.5950,Катастрофа!
2,"Исправленная (Ridge + шум, alpha = 100)",0.5960,Спасение!



 Вопросы: 
2. Насколько упало качество (R2) после добавления шума? На 0.0008 единиц.
3. При каком alpha Ridge показал лучший результат? При alpha = 100
4. Удалось ли вернуть качество к базовому уровню? Да, разрыв с базовой моделью составляет всего -0.0002. Ridge обнулил влияние 30 шумовых признаков.
